# Buổi 7: Cửa sổ và transaction
**GIẢ LẬP**
## Câu hỏi → dự đoán
Mỗi dòng kết quả đại diện cho gì? Dự đoán số dòng trước khi chạy.

In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
while not (root/'quality_lab').is_dir() and root!=root.parent: root=root.parent
if not (root/'quality_lab').is_dir(): raise RuntimeError('Mở bằng python run_quality.py --notebook tại thư mục dự án')
sys.path.insert(0,str(root))
from quality_lab.learning import read_sql,learning_db
db=learning_db()
print('Database chỉ đọc:',db)


## SQL mẫu → giải thích
LAG cho lần kiểm tra trước, nhưng giữa hai lần còn có sửa. Timeline đầy đủ cần UNION ALL ba loại sự kiện.

In [ ]:
read_sql('SELECT unit_id,inspection_no,inspected_at,LAG(inspected_at) OVER(PARTITION BY unit_id ORDER BY inspection_no) AS previous_inspection FROM inspections ORDER BY unit_id,inspection_no LIMIT 20;')

## Thử thay đổi
So cấp dữ liệu, số dòng và mẫu số của truy vấn thứ hai.

In [ ]:
read_sql('SELECT unit_id,completed_at FROM reworks ORDER BY completed_at LIMIT 10;')

## Transaction/rollback trên SQLite tạm
Kiểm tra và nhiều dòng lỗi phải cùng lưu hoặc cùng hủy.

In [ ]:
import sqlite3
c=sqlite3.connect(':memory:')
c.execute('PRAGMA foreign_keys=ON')
c.executescript('CREATE TABLE inspections(id INTEGER PRIMARY KEY); CREATE TABLE defects(id INTEGER REFERENCES inspections, type INTEGER, UNIQUE(id,type));')
try:
    with c:
        c.execute('INSERT INTO inspections VALUES(1)')
        c.execute('INSERT INTO defects VALUES(1,1)')
        c.execute('INSERT INTO defects VALUES(1,1)')
except sqlite3.IntegrityError: pass
assert c.execute('SELECT COUNT(*) FROM inspections').fetchone()[0]==0
assert c.execute('SELECT COUNT(*) FROM defects').fetchone()[0]==0
c.close()

## Tự làm → kiểm tra
Làm bài tương ứng trong docs/exercises.md trước khi đọc solutions.md. Kiểm tra số dòng, mẫu số và NULL.

In [ ]:
assert int(read_sql('SELECT COUNT(*) AS n FROM units').iloc[0,0]) >= int(read_sql('SELECT COUNT(DISTINCT unit_id) AS n FROM inspections WHERE inspection_no=1').iloc[0,0])
print('Số đã kiểm tra đầu không lớn hơn số đơn vị')